# 01 評価の基礎 — 「当たった」をどう測るか

## このノートの目的
- **正解率だけでは、まれな事象の予測を評価できない**ことを、数字で確かめる。
- **適合率・再現率・PR-AUC・ROC-AUC**の意味と使い分けを知る。
- **データの分け方**（ランダム／日単位／年単位）で、評価が大きく変わることを体験する。

## キーワード
| 用語 | やさしい説明 |
|---|---|
| **混同行列** | 予測と正解の組み合わせを数えた2×2の表。 |
| **適合率（precision）** | 「大雨と予測した」うち、**本当に大雨だった割合**。外れ（空振り）の少なさ。 |
| **再現率（recall）** | 「本当に大雨だった」うち、**予測できた割合**。見逃しの少なさ。 |
| **しきい値** | 予測確率がいくつ以上なら「大雨」と判定するか。上げると適合率↑・再現率↓。 |
| **PR-AUC** | 適合率と再現率の曲線の下の面積。**陽性が少ないときの標準的な指標**。ランダムに当てても、**陽性率**と同じ値になる。 |
| **ROC-AUC** | 0.5 が無相関、1.0 が完全な順位づけ。陽性が少なくても大きく見えやすい。 |
| **leave-one-year-out（LOYO）** | 1年を評価に回し、残りの年で学習する。これを全年で繰り返す。 |

In [ ]:
# 準備: 必要なライブラリを読み込み、src フォルダの部品（説明変数を作る dataset.py など）を使えるようにします
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("..").resolve() / "src"))   # ノートブックは notebooks/ にあるので、1つ上の src を探す
plt.rcParams["font.family"] = ["Hiragino Sans", "Arial Unicode MS", "sans-serif"]  # 日本語を表示するための設定
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.width", 140)
import dataset as ds   # 説明変数と正解を作る部品（src/dataset.py）

In [ ]:
# 目的: 以降で使う「予測スコア」を作る。ロジスティック回帰（単純な基準モデル）を、年ごとに保持して予測する（leave-one-year-out）
from sklearn.metrics import (average_precision_score, roc_auc_score, confusion_matrix, precision_recall_curve, roc_curve)
import run_phase2 as p2                      # 保持予測を作る部品（src/run_phase2.py の oof）

target = "y20_region"                        # 正解: 午後1〜6時に、地域のどこかで 20mm/h 以上
x = ds.build("naha")
f = ds.features("naha", change=False)
score = p2.oof(x, f, target, "logreg", p2.BASE, 0)             # 各標本の「大雨になる確率」（その年を学習に使わずに予測）
use = (x.year.isin(range(2010, 2025)) & x.wind_ok).to_numpy()  # 評価に使う標本（2010〜2024年、風速10m/s未満）
y, s = x.loc[use, target].to_numpy(), score[use]
print(f"評価に使う標本: {len(y)}、陽性: {y.sum()}（陽性率 {y.mean():.3f}）")

**出力の見方**: 標本のうち**約3.5%だけが陽性**です。「何も考えずに全部『大雨なし』と答える」だけで、正解率は約96.5%になります。
では、正解率で評価したらどうなるでしょう。

## 1. 正解率の落とし穴 — 「常に0」の予測

In [ ]:
# 目的: 「常に大雨なしと答える」予測の正解率・適合率・再現率を計算する
pred_zero = np.zeros_like(y)
acc = (pred_zero == y).mean()
print(f"正解率（accuracy）: {acc:.3f}")
print("混同行列（行=本当、列=予測）:\n", confusion_matrix(y, pred_zero))
print("再現率:", 0.0, "（大雨を1回も当てていない）")

**出力の見方**: 正解率は 0.96 と高いのに、**大雨を1回も当てていません**。まれな事象では、**正解率は役に立ちません**。
代わりに、次の**適合率**と**再現率**を使います。

## 2. 混同行列・適合率・再現率

予測スコアに**しきい値**を決めて、「大雨あり／なし」に変換してみます。ここでは、**スコアの上位10%を「大雨あり」**と判定します。

In [ ]:
# 目的: スコア上位10%を「大雨あり」と判定したときの、混同行列・適合率・再現率を求める
q = 10                                               # ← 演習: 5, 20, 30 に変えてみよう（上位 q % を警戒）
thr = np.percentile(s, 100 - q)                      # しきい値 = 上位 q% の境目のスコア
pred = (s >= thr).astype(int)
tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
print(f"しきい値 {thr:.3f}")
print(pd.DataFrame([[tp, fn], [fp, tn]], index=["本当は大雨あり", "本当は大雨なし"], columns=["あり と予測", "なし と予測"]))
print(f"適合率 = TP/(TP+FP) = {tp}/{tp + fp} = {tp / (tp + fp):.3f}   ← 「あり」と言ったうち当たった割合")
print(f"再現率 = TP/(TP+FN) = {tp}/{tp + fn} = {tp / (tp + fn):.3f}   ← 本当の大雨のうち拾えた割合")

**出力の見方**
- **TP**（真陽性）: 大雨と予測して本当に大雨。**FP**（偽陽性）: 大雨と予測したが降らなかった（空振り）。
- **FN**（偽陰性）: 大雨を見逃した。**TN**（真陰性）: 降らないと予測して降らなかった。
- 警戒を広げる（`q` を大きくする）と**再現率は上がり、適合率は下がります**。これが**トレードオフ**です。
  何を重視するか（見逃しを減らすか、空振りを減らすか）で、しきい値を選びます。

## 3. PR曲線とROC曲線 — しきい値を動かしながら見る

しきい値を全部動かして、適合率と再現率（**PR曲線**）、あるいは偽陽性率と再現率（**ROC曲線**）を描きます。

In [ ]:
# 目的: PR曲線とROC曲線を描き、面積（PR-AUC・ROC-AUC）を求める
pr, rc, _ = precision_recall_curve(y, s)
fpr, tpr, _ = roc_curve(y, s)
ap, auc = average_precision_score(y, s), roc_auc_score(y, s)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(rc, pr, label=f"モデル（PR-AUC={ap:.3f}）")
axes[0].axhline(y.mean(), color="gray", ls="--", label=f"ランダム（陽性率 {y.mean():.3f}）")
axes[0].set_xlabel("再現率"); axes[0].set_ylabel("適合率"); axes[0].set_title("PR曲線"); axes[0].legend()
axes[1].plot(fpr, tpr, label=f"モデル（ROC-AUC={auc:.3f}）")
axes[1].plot([0, 1], [0, 1], color="gray", ls="--", label="ランダム（0.5）")
axes[1].set_xlabel("偽陽性率"); axes[1].set_ylabel("再現率"); axes[1].set_title("ROC曲線"); axes[1].legend()
plt.tight_layout(); plt.show()
print(f"PR-AUC = {ap:.3f}（ランダムなら {y.mean():.3f}。何倍か: {ap / y.mean():.1f}倍）／ ROC-AUC = {auc:.3f}（ランダムなら 0.5）")

**出力の見方**
- **PR-AUC** は、陽性率（約0.035）が**「何も知らない場合の値」**です。0.08 なら「約2倍」と読みます。
- **ROC-AUC** は 0.5 が基準で、0.7 前後なら「そこそこ順位づけできている」ように見えます。
  しかし、陽性が少ないと、**ROC-AUC は良く見えやすい**のが欠点です。次のセルで確かめます。

In [ ]:
# 目的: 「同じ性能の予測」でも、陽性率が下がるとPR-AUCは大きく下がり、ROC-AUCはほとんど変わらないことを、人工データで確かめる
rng = np.random.default_rng(0)
rows = []
for prev in (0.5, 0.1, 0.035, 0.01):                   # 陽性率
    n = 200000
    yy = (rng.random(n) < prev).astype(int)
    ss = rng.normal(0, 1, n) + 1.0 * yy                 # 陽性ならスコアが平均で1だけ高い（性能は固定）
    rows.append({"陽性率": prev, "ROC-AUC": roc_auc_score(yy, ss), "PR-AUC": average_precision_score(yy, ss),
                 "PR-AUC ÷ 陽性率": average_precision_score(yy, ss) / prev})
pd.DataFrame(rows).round(3)

**出力の見方**: 性能（スコアの分かれ方）は同じなのに、陽性率が下がると **ROC-AUC はほぼ0.76のまま**ですが、**PR-AUC の値は 0.75 → 0.04 と大きく下がります**（陽性率の何倍か、は逆に増えます）。
陽性が少ないほど、「大雨と言ったのに降らなかった（空振り）」が増えて使いにくくなります。PR-AUC のほうが、その**現実の使いにくさ**を反映します。このプロジェクトでは PR-AUC を主な指標にし、**陽性率との比**で読みます。

## 4. データの分け方で、評価が変わる

機械学習では、**学習に使っていないデータ**で評価します。では、どう分けるのが正しいでしょうか。
3つの分け方を比べます。

1. **標本をランダムに分ける**（悪い例）
2. **日で分ける**（同じ日の3標本は同じ側に入れる）
3. **年で分ける**（leave-one-year-out。実際に使う）

> 同じ日の3標本は、説明変数も正解もほぼ同じです。ランダムに分けると、**学習データに「ほぼ同じ標本」が入ってしまい**、まるで答えを知っているような評価になります。

In [ ]:
# 目的: 同じモデルを3通りの分け方で評価して、PR-AUC の違いを見る（木を深くして「覚えやすい」設定にしてある）
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import KFold, GroupKFold

xx = x[x.year <= 2024].reset_index(drop=True)
X = xx[f].fillna(xx[f].median())
yy = xx[target]

def cv_ap(splitter, groups=None):
    pred = np.zeros(len(xx))
    for tr, te in splitter.split(X, yy, groups):
        m = ExtraTreesClassifier(n_estimators=100, min_samples_leaf=1, class_weight="balanced_subsample", n_jobs=-1, random_state=0)
        pred[te] = m.fit(X.iloc[tr], yy.iloc[tr]).predict_proba(X.iloc[te])[:, 1]
    return average_precision_score(yy, pred)

res = {"① 標本をランダムに分ける": cv_ap(KFold(5, shuffle=True, random_state=0)),
       "② 日で分ける": cv_ap(GroupKFold(5), xx.date.astype(str)),
       "③ 年で分ける（LOYO）": cv_ap(GroupKFold(15), xx.year)}
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.barh(list(res)[::-1], list(res.values())[::-1], color=["#4c72b0", "#dd8452", "#c44e52"])
ax.axvline(yy.mean(), color="gray", ls="--"); ax.set_xlabel("PR-AUC（灰色の破線は陽性率）")
plt.tight_layout(); plt.show()
print({k: round(v, 3) for k, v in res.items()})

**出力の見方**: ①のランダム分割の PR-AUC は**約0.41**と突出して高く、②日で分ける（約0.09）・③年で分ける（約0.11）は**ほぼ同じ**です。
①の高さは、**同じ日の3標本が学習側と評価側に分かれ、ほぼ同じ標本を「覚えて」当てている**ためで、**情報漏れによる見かけの高さ**です（実力の約4倍に見えています）。
②と③は同程度ですが、実際の使い方（来年の7〜9月を予測する）に一番近いのは③なので、③を採用しています。
**評価の設計を間違えると、モデルの性能を何倍にも過大評価する**ことが分かります。

## まとめ
- まれな事象では、**正解率は使えない**。**適合率・再現率・PR-AUC**を使う。
- **陽性率が基準**。PR-AUC は「陽性率の何倍か」で読む。
- 分け方は、**実際の使い方に合わせる**（ここでは年単位）。

## 演習
1. **2.** の `q = 10` を `q = 30` に変えてください（観察点: 再現率はどれだけ上がり、適合率はどれだけ下がるか）。
2. **1.** の `confusion_matrix` の結果を見て、「常に0」では `tp` と `fp` が何になるか確認してください。
3. **3.** の人工データで、`+ 1.0 * yy` を `+ 2.0 * yy`（性能を上げる）に変えてください（観察点: 陽性率が0.01でも、PR-AUC は上がるか）。
4. **4.** の `min_samples_leaf=1` を `20` に変えてください（観察点: ①の PR-AUC は下がるか。目安: 約0.41 → 約0.19。木を浅くすると「覚える」力が下がり、情報漏れの影響が減る。②③はほとんど変わらない）。